In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(cowplot)

In [ ]:
#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
first_dispense <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/First_Dispense.csv")
pheno <- pheno %>%
    left_join(first_dispense, by = "person_id")


In [ ]:
#-- Load outcomes
combo <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Combination_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
aug <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Augmentation_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_90days_noBIP_noPsychotic.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence)
  ) 

In [ ]:
# Switchers: first switch
first_switch <- outcome %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

# Continuation: first continuation (among non-switchers)
C <- outcome %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(FinalClassification.treatment == "Continuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Filter continuers to have no TRD, augmentation or combination
C_clean <- C %>%
  filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
  filter(!person_id %in% combo$person_id[combo$Case == 1])

# Discontinuation: first discontinuation (among non-switchers AND non-continuers)
ED <- outcome %>%
  filter(!person_id %in% first_switch$person_id) %>%
  filter(!person_id %in% C$person_id) %>%
  filter(FinalClassification.treatment == "Discontinuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Input data for switching vs. non-switching
dat <- rbind(first_switch, ED, C_clean) %>%
  select(person_id, FinalClassification.treatment, FirstDispense.treatment, Incidence, General_adherence_score.prior)

In [ ]:
#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))
ancd_df <- anc_df %>%
    select(research_id, ancestry_pred)
dat <- dat %>%
    left_join(anc_df, by = c("person_id" = "research_id"))

In [ ]:
#-- Join with phenotypes
final <- dat %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  )

#-- Scale age and BMI
final <- final %>%
    mutate(age_at_first_AD_unscaled = age_at_first_AD,
           age_at_first_AD = scale(age_at_first_AD),
           MDD_symptom_count_unscaled = MDD_symptom_count,
           MDD_symptom_count = scale(MDD_symptom_count),
          BMI = scale(BMI))

#-- Set reference levels
final$Education_level = relevel(as.factor(final$Education_level), ref = "Less than a high school degree")
final$Smoking_Frequency <- relevel(as.factor(final$Smoking_Frequency), ref = "Lifetime Abstainer")
final$Alcohol_Frequency = relevel(as.factor(final$Alcohol_Frequency), ref = "Lifetime Abstainer")
final$Stress = relevel(as.factor(final$Stress), ref = "Never")
final$Income = relevel(as.factor(final$Income), ref = "<=25k")
final$sex_at_birth = relevel(as.factor(final$sex_at_birth), ref = "Female")

In [ ]:
model_function <- function(input_data, independent) {
  
  print(independent)
  
  #-- Filter for complete cases
  input_data <- input_data %>%
    select(FinalClassification.treatment, age_at_first_AD, sex_at_birth, Income, Education_level,
           !!independent) %>%
    filter(complete.cases(.))
  
  #-- CRITICAL DIAGNOSTIC
  cat("\n=== DIAGNOSTIC FOR", independent, "===\n")
  cat("Number of rows:", nrow(input_data), "\n")
  cat("Unique values:", paste(unique(input_data[[independent]]), collapse=", "), "\n")
  cat("Table:\n")
  print(table(input_data[[independent]], useNA="always"))
  
  #-- Stop if no variance
  if (is.factor(input_data[[independent]])) {

    if (length(unique(input_data[[independent]])) == 1) {
      cat("NO VARIANCE (factor) - STOPPING\n")
      return("No variance")
    }

    level_counts <- table(input_data[[independent]])
    if (any(level_counts < 10)) {
      cat("SPARSE FACTOR LEVELS - STOPPING\n")
      print(level_counts)
      return("Sparse levels")
    }

    cross_tab <- table(input_data[[independent]], input_data$FinalClassification.treatment)
    if (any(cross_tab == 0)) {
      cat("COMPLETE SEPARATION RISK - some factor levels missing from an outcome\n")
      print(cross_tab)
      return("Separation risk")
    }

  } else {
    cat("Variance of", independent, ":", var(input_data[[independent]], na.rm=TRUE), "\n")
    if (var(input_data[[independent]], na.rm=TRUE) == 0) {
      cat("NO VARIANCE - STOPPING\n")
      return("No variance")
    }
  }

  #-- Check if any data remains after filtering
  if (nrow(input_data) == 0) {
    message("No complete cases remaining")
    return("Incomplete")
  }
  
  #-- Outcome prevalence
  outcome_counts <- input_data %>%
    group_by(FinalClassification.treatment) %>%
    summarise(n = n(), .groups = "drop")
  
  n_continuation <- outcome_counts %>% filter(FinalClassification.treatment == "Continuation") %>% pull(n)
  n_ed           <- outcome_counts %>% filter(FinalClassification.treatment == "Discontinuation") %>% pull(n)
  n_switching    <- outcome_counts %>% filter(FinalClassification.treatment == "Switching") %>% pull(n)
  
  if (length(n_continuation) == 0) n_continuation <- 0
  if (length(n_ed) == 0) n_ed <- 0
  if (length(n_switching) == 0) n_switching <- 0
  
  n_total <- nrow(input_data)
  
  #-- Skip if any outcome has less than 20 cases
  if (n_continuation < 20 | n_ed < 20 | n_switching < 20) {
    message(sprintf("Insufficient cases: Cont=%d, D=%d, Switch=%d", n_continuation, n_ed, n_switching))
    return("Incomplete")
  }
  
  #-- Build formula based on predictor type
  glm_formula <- if (independent %in% c("age_at_first_AD", "sex_at_birth")) {
    as.formula("outcome_binary ~ age_at_first_AD + sex_at_birth")
  } else if (independent %in% c("Income", "Education_level")) {
    as.formula("outcome_binary ~ age_at_first_AD + sex_at_birth + Income + Education_level")
  } else {
    as.formula(paste("outcome_binary ~ age_at_first_AD + sex_at_birth + Income + Education_level +", independent))
  }
  
  #-- Helper to run one binary logistic regression and extract results
  run_glm <- function(data, comparison_label) {
    
    # Check separation on the binary subset actually being modelled
    for (fac_var in c("Income", "Education_level")) {
      if (fac_var %in% names(data) && is.factor(data[[fac_var]])) {
        cross_tab <- table(data[[fac_var]], data$outcome_binary)
        if (any(cross_tab == 0)) {
          cat("SEPARATION in", comparison_label, "for", fac_var, "- SKIPPING\n")
          print(cross_tab)
          return(NULL)
        }
      }
    }
    
    model  <- glm(glm_formula, data = data, family = binomial())
    
    coefs  <- coef(model)
    ses    <- sqrt(diag(vcov(model)))
    lower  <- coefs - 1.96 * ses
    upper  <- coefs + 1.96 * ses
    p_vals <- summary(model)$coefficients[, 4]
    
    #-- Ns from the binary subset actually modelled
    n_ref  <- sum(data$outcome_binary == 0)
    n_case <- sum(data$outcome_binary == 1)
    
    data.frame(
      outcome_comparison = comparison_label,
      variable           = names(coefs),
      coef               = coefs,
      odds_ratio         = exp(coefs),
      se                 = ses,
      z                  = coefs / ses,
      p_value            = p_vals,
      lower_95           = exp(lower),
      upper_95           = exp(upper),
      N_Total            = n_ref + n_case,
      N_Reference        = n_ref,
      N_Case             = n_case,
      stringsAsFactors   = FALSE,
      row.names          = NULL
    )
  }
  
  #-- Switching vs Continuation
  data_sw <- input_data %>%
    filter(FinalClassification.treatment %in% c("Continuation", "Switching")) %>%
    mutate(outcome_binary = as.integer(FinalClassification.treatment == "Switching"))
  
  #-- Discontinuation vs Continuation
  data_ed <- input_data %>%
    filter(FinalClassification.treatment %in% c("Continuation", "Discontinuation")) %>%
    mutate(outcome_binary = as.integer(FinalClassification.treatment == "Discontinuation"))
  
  res_sw <- run_glm(data_sw, "Switching vs Continuation")
  res_ed <- run_glm(data_ed, "Discontinuation vs Continuation")
  results <- bind_rows(Filter(Negate(is.null), list(res_sw, res_ed)))
  
  if (nrow(results) == 0) return("Separation risk")
  
  return(results)
}

In [ ]:
#-- Select phenotypes to be associated with 
predictors <- c("Anxiety", "Depression", "Smoking_Frequency", "Income", "BMI", "Migraine", "Insomnia",
                "Stress", "Education_level", "Alcohol_Frequency", "age_at_first_AD", "sex_at_birth", "MDD_symptom_count", "MDD_Incl")

#-- Load Switching Classifications and loop through each condition combination
population_groups <- c("afr", "eur", "amr")
clinical_groups = c("Full", "MDD", "noMDD", "ANX", "CIDI-MDD", "CIDI-noMDD")
all_list <- list() 
    
for (clinical in clinical_groups) {
  
  cat("Clinical:", clinical, "\n")
    
  #== Filter for clinical group of interest
    if (clinical == "MDD") {
      input_data <- final %>%
        filter(Depression == 1)
    } else if (clinical == "noMDD") {
      input_data <- final %>%
        filter(Depression == 0)
    } else if (clinical == "ANX") {
        input_data <- final %>%
            filter(Anxiety == 1)
    } else if (clinical == "CIDI-MDD") {
        input_data <- final %>%
            filter(MDD_Incl == 1)
    }  else if (clinical == "CIDI-noMDD") {
        input_data <- final %>%
            filter(MDD_Incl == 0)
      } else {
        input_data <- final
    }
  
  #== Initialize list to store results across populations
  clinical_results_list <- list()
    
  #== Filter for population group
  for (population in population_groups) {
      
    cat("Ancestry:", population, "\n")
  
    if (population == "all") {
        input_data_anc <- input_data
    } else {
       input_data_anc <- input_data %>%
          filter(ancestry_pred == population) 
    }
    
    results_list <- list()
    
    #== Loop through each predictor
    for (independent in predictors){
      
      if (independent %in% c("Depression", "MDD_Incl") & (clinical %in% c("CIDI-MDD", "CIDI-noMDD"))) next
      multinom_results <- tryCatch(
        {
          model_function(input_data_anc, independent)
        },
        error = function(e) {
          message("Error for ", independent, ": ", e$message)
          return("Error")
        }
      )
    
      if (!is.data.frame(multinom_results)) next
    
      multinom_results <- multinom_results %>%
        mutate(
          GeneticMarker = independent,
          ClinicalGroup = clinical,
          Population = population
        )
      results_list[[independent]] <- multinom_results
    }
    
    #== Store population results
    if (length(results_list) > 0) {
      pop_results_df <- do.call(rbind, results_list)
      clinical_results_list[[population]] <- pop_results_df
    }
  }
  
  #== Combine results across all populations for this clinical group
  if (length(clinical_results_list) > 0) {
    results <- do.call(rbind, clinical_results_list)
    rownames(results) <- NULL
    
    #== Multiple testing correction
    results_format <- results %>%
      select(Population, ClinicalGroup, outcome_comparison, GeneticMarker, 
             N_Total, N_Reference, N_Case, 
             variable, odds_ratio, lower_95, upper_95, p_value)

    results_sig <- results_format  %>%
      group_by(outcome_comparison, Population, ClinicalGroup) %>%
      mutate(
        Bonf_P = pmin(p_value * 14, 1)
          ) %>%
      ungroup()
    
    #== Format results
    results_format_sig_rounded <- results_sig %>%
      mutate(
        across(c(odds_ratio, lower_95, upper_95), ~ signif(.x, 4)),
        p_value_num = p_value,
        Bonf_P_num  = Bonf_P,
        p_value_display = case_when(
          p_value < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(p_value, format = "e", digits = 2)
        ),
        Bonf_P_display = case_when(
          Bonf_P < 2.2e-16 ~ "<2.2e-16",
          TRUE ~ formatC(Bonf_P, format = "e", digits = 2)
        )
      )
    all_list[[clinical]] <- results_format_sig_rounded
  }
}

In [ ]:
results_df <- bind_rows(all_list) %>%
    mutate(Population = case_when(
    Population == "afr" ~ "African",
    Population == "eur" ~ "European",
    Population == "amr" ~ "Latino/Mixed-American",
    TRUE ~ NA_character_)) %>%
    rename(Phenotype = GeneticMarker) %>%
    mutate(ClinicalGroup = case_when(
    ClinicalGroup == "Full" ~ "Full Cohort",
    ClinicalGroup == "MDD" ~ "Recorded Depression",
    ClinicalGroup == "ANX" ~ "Recorded Anxiety",
    ClinicalGroup == "noMDD" ~ "No Recorded Depression",
    TRUE ~ ClinicalGroup),
          Phenotype = case_when(
          Phenotype == "Depression" ~ "Recorded Depression",
          Phenotype == "Anxiety" ~ "Recorded Anxiety",
          Phenotype == "MDD_Incl" ~ "MDD CIDI-sf",
          TRUE ~ Phenotype),
          variable = case_when(
          variable == "Depression" ~ "Recorded Depression",
          variable == "Anxiety" ~ "Recorded Anxiety",
          variable == "MDD_Incl" ~ "MDD CIDI-sf",
          TRUE ~ variable)) 

pop_order <- c("European", "African", "Latino/Mixed-American")

clinical_order <- c(
  "Full Cohort",
  "Recorded Depression",
  "No Recorded Depression",
  "Recorded Anxiety",
  "CIDI-MDD",
  "CIDI-noMDD"
)

results_df <- results_df %>%
  rename(Dependent = Phenotype,
        Outcome = outcome_comparison) %>%
  mutate(
    Population = factor(Population, levels = pop_order),
    ClinicalGroup = factor(ClinicalGroup, levels = clinical_order)
  ) %>%
  arrange(Population, ClinicalGroup, Dependent, Outcome)

results_df %>% filter(odds_ratio > 10)

In [ ]:
path <- "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx"

if (file.exists(path)) {
  wb <- loadWorkbook(path)
} else {
  wb <- createWorkbook()
}

# removeWorksheet only makes sense if the sheet is already there
if ("Table2" %in% names(wb)) {
  removeWorksheet(wb, "Table2")
}
addWorksheet(wb, "Table2")
writeData(wb, "Table2", results_df)

saveWorkbook(wb, path, overwrite = TRUE)

In [ ]:
order <- c(
  # Mood / mental health (binary)
  "MDD CIDI-sf",
  "Recorded Anxiety",
  # Non-mood conditions (binary)
  "Insomnia",
  "Migraine",
  # Quantitative
  "BMI",
  "age_at_first_AD",
  "MDD_symptom_count",
  # Sex
  "SexMale",
  # Smoking frequency (former → daily)
  "Not At All (former)",
  "Some Days",
  "Every Day",
  # Stress (rare → frequent)
  "Almost Never",
  "Sometimes",
  "Fairly Often",
  "Very Often",
  # Income (low → high)   [ref = <=25k]
  "25-35k",
  "35-50k",
  "50-75k",
  "75-100k",
  "100-150k",
  "150-200k",
  ">200k",
  # Education (low → high)   [ref = Less than a high school degree]
  "Twelve or GED",
  "College Undergraduate",
  "College Postgraduate",
  # Drink frequency (former → frequent)
  "Never (former)",
  "Monthly or Less",
  "2-4 Month",
  "2-3 Week",
  "4plus Week"
)

In [ ]:
## Aou: Europeans with MDD
library(readxl)
library(dplyr)
library(tidyverse)

phenotypes_of_interest <- c("Recorded Anxiety", "Recorded Depression", "Smoking_Frequency","Migraine", "Insomnia", "BMI",
                            "Income", "Stress", "Education_level", "MDD CIDI-sf", "MDD_symptom_count",
                            "Alcohol_Frequency", "age_at_first_AD", "sex_at_birth")

#-- Multinomial regression plot (Pooled within all EUR)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table2")
processed_data %>% filter(Population == "European") %>% nrow()
processed_data %>% filter(Population == "European", ClinicalGroup == "Recorded Depression") %>% nrow()
processed_data <- processed_data %>%
  filter(Population == "European",
         ClinicalGroup == "Recorded Depression",
         variable != "(Intercept)") %>%
  filter(str_starts(variable, Dependent) | variable == Dependent) %>%
  mutate(Outcome = ifelse(Outcome == "Switching vs Continuation",
                          "Switching", "Discontinuation"))

#=== Clean variable names
processed_data <- processed_data %>%
    mutate(variable = gsub("Smoking_Frequency", "", variable)) %>%
    mutate(variable = gsub("Stress", "", variable)) %>%
    mutate(variable = gsub("Alcohol_Frequency", "", variable)) %>%
    mutate(variable = gsub("Education_level", "", variable)) %>%
    mutate(variable = gsub("Income", "", variable)) %>%
    mutate(variable = gsub("sex_at_birth", "Sex", variable)) %>%
    mutate(fill_color = if_else(Bonf_P_num < 0.05, Outcome, "Non-Sig"))

processed_data <- processed_data %>%
    mutate(Dependent = case_when(
    Dependent %in% c("Recorded Depression", "MDD CIDI-sf", "Recorded Anxiety") ~ "Mood Disorders\n(Binary)",
    Dependent %in% c("Insomnia", "Migraine") ~ "Non-Mood\nConditions (Binary)",
    Dependent == "Education_level" ~ "Education Level\n(ref=less than HS)",
        Dependent == "Smoking_Frequency" ~ "Smoking Frequency\n(ref=abstainer)",
        Dependent == "Alcohol_Frequency" ~ "Alcohol Frequency\n(ref=abstainer)",
        Dependent == "Stress" ~ "Stress\n(ref=never)",
        Dependent == "Income" ~ "Income\n(ref=<=25k)",
        Dependent == "sex_at_birth" ~ "Sex\n(ref=female)",
        Dependent %in% c("age_at_first_AD", "BMI", "MDD_symptom_count") ~ "Quantitative\n(scaled)",
    TRUE ~ Dependent))

processed_data$variable <- factor(processed_data$variable, levels = order)
levels(processed_data$variable)[levels(processed_data$variable) == "age_at_first_AD"] <- "Age at First AD"
levels(processed_data$variable)[levels(processed_data$variable) == "MDD_symptom_count"] <- "MDD CIDI-sf\nsymptom count"

processed_data$Dependent <- factor(
    processed_data$Dependent, 
    levels = c("Mood Disorders\n(Binary)", "Non-Mood\nConditions (Binary)", "Quantitative\n(scaled)", "Sex\n(ref=female)", 
               "Smoking Frequency\n(ref=abstainer)", "Income\n(ref=<=25k)", "Education Level\n(ref=less than HS)", "Stress\n(ref=never)",
               "Alcohol Frequency\n(ref=abstainer)"))
# Extract the colors first
my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29" 
)
names(my_colors) <- c("Discontinuation", "Switching")

#-- Results Pooled within EUR in Lifelines with MDD
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL_noANX.xlsx", sheet = "Table1") %>%
  filter(ClinicalGroup == "Recorded Depression") %>%
  rename(Phenotype = Dependent) %>%
  filter(!variable %in% c("(Intercept)")) %>%
  filter(!(!variable %in% c("age_at_first_AD") & Phenotype %in% c("age_at_first_AD"))) %>%
  filter(!(!variable %in% c("sex_at_birthMale") & Phenotype %in% c("sex_at_birth"))) %>%
  filter(!(!variable %in% c("Income_level") & Phenotype %in% c("Income_level"))) %>%
  filter(!(!variable %in% c("Education_level") & Phenotype %in% c("Education_level"))) %>%
  filter(!(variable == "age_at_first_AD" & Phenotype == "sex_at_birth")) %>%
  filter(!(variable %in% c("sex_at_birthMale", "age_at_first_AD", "Education_level", "Income_level") & !Phenotype %in% c("sex_at_birth", "age_at_first_AD", "Education_level", "Income_level"))) %>%
    mutate(Outcome = ifelse(Outcome == "Switching vs Continuation", "Switching", "Discontinuation"))

dat <- dat %>%
  mutate(
    odds_ratio = as.numeric(odds_ratio),
    lower_95 = as.numeric(lower_95),
    upper_95 = as.numeric(upper_95),
    fill_color=if_else(Bonf_P < 0.05, Outcome, "NonSig")
  )


my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29"
)
names(my_colors) <- c("Discontinuation", "Switching")

dat <- dat %>%
  mutate(
    Phenotype =
      case_when(
        Phenotype == "sex_at_birth" ~ "Sex (ref=female)",
        TRUE ~ Phenotype
      )
  )

dat$Phenotype <- factor(dat$Phenotype, levels =
                          rev(c("age_at_first_AD",
                            "Sex (ref=female)",
                            "Education_level",
                            "BMI",
                            "Income_level",
                            "Depression",
                            "Lifetime_anxiety",
                            "Neuroticism",
                            "Chronotype",
                            "Sleep_weekly",
                            "Childhood_sexual_abuse",
                            "Threatening_experiences",
                            "Longterm_difficulties",
                            "Alcohol_frequency",
                            "Smoking_Packyears",
                            "Fibromyalgia")))

dat = dat %>%
  mutate(
    Type = case_when(
    !Phenotype %in% c("Lifetime_anxiety", "Fibromyalgia", "Sex (ref=female)") ~ "Quantitative Traits\n(scaled)",
    Phenotype %in% c("Lifetime_anxiety", "Fibromyalgia", "Sex (ref=female)") ~ "Binary Traits",
    TRUE ~ NA_character_)
  )


#levels(dat$Phenotype) <- gsub("_", " ", levels(dat$Phenotype))
# Or for finer control:
levels(dat$Phenotype) <- recode(levels(dat$Phenotype),
  "Lifetime_anxiety"        = "Lifetime Anxiety",
  "Childhood_sexual_abuse"  = "Childhood Sexual Abuse",
  "Threatening_experiences" = "Threatening Experiences",
  "Longterm_difficulties"   = "Long-term Difficulties",
  "Alcohol_frequency"       = "Alcohol Intake",
  "Smoking_Packyears"       = "Smoking Pack-years",
  "Sleep_weekly"            = "Sleep (Weekly)",
  "Education_level"         = "Education Level",
  "Income_level"            = "Income Level",
  "age_at_first_AD"         = "Age at First AD"
)

pos <- position_dodge(width = 0.5)

point_layer <- list(
  geom_vline(xintercept = 1, linetype = "dashed",
             linewidth = 0.5, color = "#333333"),
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 linewidth = 0.3, height = 0, position = pos),
  geom_point(shape = 21, size = 3, stroke = 0.5, position = pos)
)

#-- AoU panel
p_multi <- ggplot(processed_data,
                  aes(y = variable, x = odds_ratio,
                      color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "All of Us",
       color = "Outcome",
      subtitle = "Participants with Recorded Depression",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "Non-Sig" = "white")) +
  facet_wrap(~Dependent, scales = "free", ncol = 2, strip.position = "left") +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

#-- LL panel (same layers, different data + facet)
p <- ggplot(dat,
            aes(y = Phenotype, x = odds_ratio,
                color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "Lifelines",
       color = "Outcome",
      subtitle = "Participants with\nRecorded Depression",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "NonSig" = "white")) +
  facet_wrap(~ Type, scales = "free", strip.position = "left", nrow = 2) +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold"),
    panel.grid.minor = element_blank()
  )

all <- plot_grid(p_multi, (p + theme(plot.margin = margin(t = 5, r = 5, b = 120, l = 5))), 
                 nrow = 1, labels = c("A", "B"), rel_widths = c(1.3, 1))
all

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Main_Pooled_Phenotypic_plot_MDD.png",
       plot   = all,
       width  = 10,
       height = 10,
       device = "png",
       units  = "in")

In [ ]:
## Aou: Europeans without MDD
library(readxl)
library(dplyr)
library(tidyverse)

phenotypes_of_interest <- c("Recorded Anxiety", "Recorded Depression", "Smoking_Frequency","Migraine", "Insomnia", "BMI",
                            "Income", "Stress", "Education_level", "MDD CIDI-sf", "MDD_symptom_count",
                            "Alcohol_Frequency", "age_at_first_AD", "sex_at_birth")

#-- Multinomial regression plot (Pooled within all EUR)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table2")
processed_data <- processed_data %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup == "No Recorded Depression") %>%
  filter(variable != "(Intercept)") %>%
  filter(
    str_detect(variable, paste0("^", Dependent)) | 
    variable == Dependent
  ) %>%
    mutate(Outcome = ifelse(Outcome == "Switching vs Continuation", "Switching", "Discontinuation"))

#=== Clean variable names
processed_data <- processed_data %>%
    mutate(variable = gsub("Smoking_Frequency", "", variable)) %>%
    mutate(variable = gsub("Stress", "", variable)) %>%
    mutate(variable = gsub("Alcohol_Frequency", "", variable)) %>%
    mutate(variable = gsub("Education_level", "", variable)) %>%
    mutate(variable = gsub("Income", "", variable)) %>%
    mutate(variable = gsub("sex_at_birth", "Sex", variable)) %>%
    mutate(fill_color = if_else(Bonf_P_num < 0.05, Outcome, "Non-Sig"))


processed_data <- processed_data %>%
    mutate(Dependent = case_when(
    Dependent %in% c("Recorded Depression", "MDD CIDI-sf", "Recorded Anxiety") ~ "Mood Disorders\n(Binary)",
    Dependent %in% c("Insomnia", "Migraine") ~ "Non-Mood\nConditions (Binary)",
    Dependent == "Education_level" ~ "Education Level\n(ref=less than HS)",
        Dependent == "Smoking_Frequency" ~ "Smoking Frequency\n(ref=abstainer)",
        Dependent == "Alcohol_Frequency" ~ "Alcohol Frequency\n(ref=abstainer)",
        Dependent == "Stress" ~ "Stress\n(ref=never)",
        Dependent == "Income" ~ "Income\n(ref=<=25k)",
        Dependent == "sex_at_birth" ~ "Sex\n(ref=female)",
        Dependent %in% c("age_at_first_AD", "BMI", "MDD_symptom_count") ~ "Quantitative\n(scaled)",
    TRUE ~ Dependent))

processed_data$variable <- factor(processed_data$variable, levels = order)
levels(processed_data$variable)[levels(processed_data$variable) == "age_at_first_AD"] <- "Age at First AD"
levels(processed_data$variable)[levels(processed_data$variable) == "MDD_symptom_count"] <- "MDD CIDI-sf\nsymptom count"

processed_data$Dependent <- factor(
    processed_data$Dependent, 
    levels = c("Mood Disorders\n(Binary)", "Non-Mood\nConditions (Binary)", "Quantitative\n(scaled)", "Sex\n(ref=female)", 
               "Smoking Frequency\n(ref=abstainer)", "Income\n(ref=<=25k)", "Education Level\n(ref=less than HS)", "Stress\n(ref=never)",
               "Alcohol Frequency\n(ref=abstainer)"))

# Extract the colors first
my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29" 
)
names(my_colors) <- c("Discontinuation", "Switching")

#-- Results Pooled within EUR in Lifelines with MDD
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL_noANX.xlsx", sheet = "Table1") %>%
  filter(ClinicalGroup == "No Recorded Depression") %>%
  rename(Phenotype = Dependent) %>%
  filter(!variable %in% c("(Intercept)")) %>%
  filter(!(!variable %in% c("age_at_first_AD") & Phenotype %in% c("age_at_first_AD"))) %>%
  filter(!(!variable %in% c("sex_at_birthMale") & Phenotype %in% c("sex_at_birth"))) %>%
  filter(!(!variable %in% c("Income_level") & Phenotype %in% c("Income_level"))) %>%
  filter(!(!variable %in% c("Education_level") & Phenotype %in% c("Education_level"))) %>%
  filter(!(variable == "age_at_first_AD" & Phenotype == "sex_at_birth")) %>%
  filter(!(variable %in% c("sex_at_birthMale", "age_at_first_AD", "Education_level", "Income_level") & !Phenotype %in% c("sex_at_birth", "age_at_first_AD", "Education_level", "Income_level"))) %>%
    mutate(Outcome = ifelse(Outcome == "Switching vs Continuation", "Switching", "Discontinuation"))

dat <- dat %>%
  mutate(
    odds_ratio = as.numeric(odds_ratio),
    lower_95 = as.numeric(lower_95),
    upper_95 = as.numeric(upper_95),
    fill_color=if_else(Bonf_P < 0.05, Outcome, "NonSig")
  )


my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29"
)
names(my_colors) <- c("Discontinuation", "Switching")

dat <- dat %>%
  mutate(
    Phenotype =
      case_when(
        Phenotype == "sex_at_birth" ~ "Sex (ref=female)",
        TRUE ~ Phenotype
      )
  )

dat$Phenotype <- factor(dat$Phenotype, levels =
                          rev(c("age_at_first_AD",
                            "Sex (ref=female)",
                            "Education_level",
                            "BMI",
                            "Income_level",
                            "Depression",
                            "Lifetime_anxiety",
                            "Neuroticism",
                            "Chronotype",
                            "Sleep_weekly",
                            "Childhood_sexual_abuse",
                            "Threatening_experiences",
                            "Longterm_difficulties",
                            "Alcohol_frequency",
                            "Smoking_Packyears",
                            "Fibromyalgia")))

dat = dat %>%
  mutate(
    Type = case_when(
    !Phenotype %in% c("Lifetime_anxiety", "Fibromyalgia", "Sex (ref=female)") ~ "Quantitative Traits\n(scaled)",
    Phenotype %in% c("Lifetime_anxiety", "Fibromyalgia", "Sex (ref=female)") ~ "Binary Traits",
    TRUE ~ NA_character_)
  )


# Or for finer control:
levels(dat$Phenotype) <- recode(levels(dat$Phenotype),
  "Lifetime_anxiety"        = "Lifetime Anxiety",
  "Childhood_sexual_abuse"  = "Childhood Sexual Abuse",
  "Threatening_experiences" = "Threatening Experiences",
  "Longterm_difficulties"   = "Long-term Difficulties",
  "Alcohol_frequency"       = "Alcohol Intake",
  "Smoking_Packyears"       = "Smoking Pack-years",
  "Sleep_weekly"            = "Sleep (Weekly)",
  "Education_level"         = "Education Level",
  "Income_level"            = "Income Level",
  "age_at_first_AD"         = "Age at First AD"
)

pos <- position_dodge(width = 0.5)

point_layer <- list(
  geom_vline(xintercept = 1, linetype = "dashed",
             linewidth = 0.5, color = "#333333"),
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 linewidth = 0.3, height = 0, position = pos),
  geom_point(shape = 21, size = 3, stroke = 0.5, position = pos)
)

#-- AoU panel
p_multi <- ggplot(processed_data,
                  aes(y = variable, x = odds_ratio,
                      color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "All of Us",
       color = "Outcome",
      subtitle = "Participants without Recorded Depression",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "Non-Sig" = "white")) +
  facet_wrap(~Dependent, scales = "free", ncol = 2, strip.position = "left") +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 10, color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold", size = 14),
    plot.subtitle    = element_text(size = 12),
    panel.grid.minor = element_blank()
  )

#-- LL panel (same layers, different data + facet)
p <- ggplot(dat,
            aes(y = Phenotype, x = odds_ratio,
                color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "Lifelines",
       color = "Outcome",
      subtitle = "Participants without\nRecorded Depression",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "NonSig" = "white")) +
  facet_wrap(~ Type, scales = "free", strip.position = "left", nrow = 2) +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 10, color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold", size = 14),
    plot.subtitle    = element_text(size = 12),
    panel.grid.minor = element_blank()
  )
library(cowplot)
all <- plot_grid(p_multi, (p + theme(plot.margin = margin(t = 5, r = 5, b = 120, l = 5))), 
                 nrow = 1, labels = c("A", "B"), rel_widths = c(1.3, 1))
all

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Pooled_Phenotypic_plot_noMDD.png",
       plot   = all,
       width  = 10,
       height = 10,
       device = "png",
       units  = "in")

In [ ]:
## Aou: Europeans with CIDI MDD
library(readxl)
library(dplyr)
library(tidyverse)

phenotypes_of_interest <- c("Recorded Anxiety", "Recorded Depression", "Smoking_Frequency", "Migraine", "Insomnia", "BMI",
                            "Income", "Stress", "Education_level", "MDD_symptom_count",
                            "Alcohol_Frequency", "age_at_first_AD", "sex_at_birth")

#-- Multinomial regression plot (Pooled within all EUR)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table2")
processed_data <- processed_data %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup == "CIDI-MDD") %>%
  filter(variable != "(Intercept)") %>%
  filter(
    str_detect(variable, paste0("^", Dependent)) | 
    variable == Dependent
  ) %>%
    mutate(Outcome = ifelse(Outcome == "Switching vs Continuation", "Switching", "Discontinuation"))

#=== Clean variable names
processed_data <- processed_data %>%
    mutate(variable = gsub("Smoking_Frequency", "", variable)) %>%
    mutate(variable = gsub("Stress", "", variable)) %>%
    mutate(variable = gsub("Alcohol_Frequency", "", variable)) %>%
    mutate(variable = gsub("Education_level", "", variable)) %>%
    mutate(variable = gsub("Income", "", variable)) %>%
    mutate(variable = gsub("sex_at_birth", "Sex", variable)) %>%
    mutate(fill_color = if_else(Bonf_P_num < 0.05, Outcome, "Non-Sig"))


processed_data <- processed_data %>%
    mutate(Dependent = case_when(
    Dependent %in% c("Recorded Depression", "Recorded Anxiety") ~ "Mood Disorders\n(Binary)",
    Dependent %in% c("Insomnia", "Migraine") ~ "Non-Mood\nConditions (Binary)",
    Dependent == "Education_level" ~ "Education Level\n(ref=less than HS)",
        Dependent == "Smoking_Frequency" ~ "Smoking Frequency\n(ref=abstainer)",
        Dependent == "Alcohol_Frequency" ~ "Alcohol Frequency\n(ref=abstainer)",
        Dependent == "Stress" ~ "Stress\n(ref=never)",
        Dependent == "Income" ~ "Income\n(ref=<=25k)",
        Dependent == "sex_at_birth" ~ "Sex\n(ref=female)",
        Dependent %in% c("age_at_first_AD", "BMI", "MDD_symptom_count") ~ "Quantitative\n(scaled)",
    TRUE ~ Dependent))


processed_data$variable <- factor(processed_data$variable, levels = order)
levels(processed_data$variable)[levels(processed_data$variable) == "age_at_first_AD"] <- "Age at First AD"
levels(processed_data$variable)[levels(processed_data$variable) == "MDD_symptom_count"] <- "MDD CIDI-sf\nsymptom count"

processed_data$Dependent <- factor(
    processed_data$Dependent, 
    levels = c("Mood Disorders\n(Binary)", "Non-Mood\nConditions (Binary)", "Quantitative\n(scaled)", "Sex\n(ref=female)", 
               "Smoking Frequency\n(ref=abstainer)", "Income\n(ref=<=25k)", "Education Level\n(ref=less than HS)", "Stress\n(ref=never)",
               "Alcohol Frequency\n(ref=abstainer)"))

# Extract the colors first
my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29" 
)
names(my_colors) <- c("Discontinuation", "Switching")

point_layer <- list(
  geom_vline(xintercept = 1, linetype = "dashed",
             linewidth = 0.5, color = "#333333"),
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 linewidth = 0.3, height = 0, position = pos),
  geom_point(shape = 21, size = 4, stroke = 0.5, position = pos)
)


#-- AoU panel
p_multi <- ggplot(processed_data,
                  aes(y = variable, x = odds_ratio,
                      color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "All of Us",
       color = "Outcome",
      subtitle = "Participants with MDD (CIDI-sf)",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "Non-Sig" = "white")) +
  facet_wrap(~Dependent, scales = "free", ncol = 2, strip.position = "left") +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 12, color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold", size = 16),
    plot.subtitle    = element_text(size = 12),
    panel.grid.minor = element_blank()
  )
p_multi

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Pooled_Phenotypic_plot_CIDI_MDD.png",
       plot   = p_multi,
       width  = 10,
       height = 10,
       device = "png",
       units  = "in")

In [ ]:
## Aou: Europeans with no CIDI MDD
library(readxl)
library(dplyr)
library(tidyverse)

phenotypes_of_interest <- c("Recorded Anxiety", "Recorded Depression", "Smoking_Frequency", "Migraine", "Insomnia", "BMI",
                            "Income", "Stress", "Education_level", "MDD_symptom_count",
                            "Alcohol_Frequency", "age_at_first_AD", "sex_at_birth")

#-- Multinomial regression plot (Pooled within all EUR)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS.xlsx", sheet = "Table2")
processed_data <- processed_data %>%
  filter(Population == "European") %>%
  filter(ClinicalGroup == "CIDI-noMDD") %>%
  filter(variable != "(Intercept)") %>%
  filter(
    str_detect(variable, paste0("^", Dependent)) | 
    variable == Dependent
  ) %>%
    mutate(Outcome = ifelse(Outcome == "Switching vs Continuation", "Switching", "Discontinuation"))

#=== Clean variable names
processed_data <- processed_data %>%
    mutate(variable = gsub("Smoking_Frequency", "", variable)) %>%
    mutate(variable = gsub("Stress", "", variable)) %>%
    mutate(variable = gsub("Alcohol_Frequency", "", variable)) %>%
    mutate(variable = gsub("Education_level", "", variable)) %>%
    mutate(variable = gsub("Income", "", variable)) %>%
    mutate(variable = gsub("sex_at_birth", "Sex", variable)) %>%
    mutate(fill_color = if_else(Bonf_P_num < 0.05, Outcome, "Non-Sig"))

processed_data <- processed_data %>%
    mutate(Dependent = case_when(
    Dependent %in% c("Recorded Depression", "Recorded Anxiety") ~ "Mood Disorders\n(Binary)",
    Dependent %in% c("Insomnia", "Migraine") ~ "Non-Mood\nConditions (Binary)",
    Dependent == "Education_level" ~ "Education Level\n(ref=less than HS)",
        Dependent == "Smoking_Frequency" ~ "Smoking Frequency\n(ref=abstainer)",
        Dependent == "Alcohol_Frequency" ~ "Alcohol Frequency\n(ref=abstainer)",
        Dependent == "Stress" ~ "Stress\n(ref=never)",
        Dependent == "Income" ~ "Income\n(ref=<=25k)",
        Dependent == "sex_at_birth" ~ "Sex\n(ref=female)",
        Dependent %in% c("age_at_first_AD", "BMI", "MDD_symptom_count") ~ "Quantitative\n(scaled)",
    TRUE ~ Dependent))

processed_data$variable <- factor(processed_data$variable, levels = order)
levels(processed_data$variable)[levels(processed_data$variable) == "age_at_first_AD"] <- "Age at First AD"
levels(processed_data$variable)[levels(processed_data$variable) == "MDD_symptom_count"] <- "MDD CIDI-sf\nsymptom count"

processed_data$Dependent <- factor(
    processed_data$Dependent, 
    levels = c("Mood Disorders\n(Binary)", "Non-Mood\nConditions (Binary)", "Quantitative\n(scaled)", "Sex\n(ref=female)", 
               "Smoking Frequency\n(ref=abstainer)", "Income\n(ref=<=25k)", "Education Level\n(ref=less than HS)", "Stress\n(ref=never)",
               "Alcohol Frequency\n(ref=abstainer)"))

# Extract the colors first
my_colors <- c(
  "Discontinuation" = "#3454D1",
  "Switching" = "#F58F29" 
)
names(my_colors) <- c("Discontinuation", "Switching")

point_layer <- list(
  geom_vline(xintercept = 1, linetype = "dashed",
             linewidth = 0.5, color = "#333333"),
  geom_errorbarh(aes(xmin = lower_95, xmax = upper_95),
                 linewidth = 0.3, height = 0, position = pos),
  geom_point(shape = 21, size = 4, stroke = 0.5, position = pos)
)

#-- AoU panel
p_multi <- ggplot(processed_data,
                  aes(y = variable, x = odds_ratio,
                      color = Outcome, fill = fill_color, group = Outcome)) +
  point_layer +
  labs(y = "Phenotype",
       title = "All of Us",
       color = "Outcome",
      subtitle = "Participants without MDD (CIDI-sf)",
      x = "Odds ratio with 95% CI,\nRef = Continuation") +
  scale_color_manual(values = my_colors) +
  scale_fill_manual(values = c(my_colors, "Non-Sig" = "white")) +
  facet_wrap(~Dependent, scales = "free", ncol = 2, strip.position = "left") +
  guides(color = guide_legend(nrow = 2, byrow = TRUE, order = 1), fill = "none") +
  theme_minimal() +
  theme(
    legend.position  = "bottom",
    axis.text        = element_text(color = "black"),
    axis.text.y      = element_text(size = 12, color = "black"),
    legend.title     = element_text(face = "bold"),
    strip.text       = element_text(face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    panel.background = element_rect(fill = "white", colour = NA),
    plot.background  = element_rect(fill = "white", colour = NA),
    plot.title       = element_text(face = "bold", size = 16),
    plot.subtitle    = element_text(size = 12),
    panel.grid.minor = element_blank()
  )
p_multi

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Pooled_Phenotypic_plot_noCIDI_MDD.png",
       plot   = p_multi,
       width  = 10,
       height = 10,
       device = "png",
       units  = "in")

In [ ]:
#-- Time to incidence and correlation with general adherence scores prior to outcome for first prescribed AD
library(lubridate)
df <- final %>%
    mutate(
    Time2Incidence = interval(DateFirstDispense, Incidence) / years(1))

test <- df %>%
    group_by(FinalClassification.treatment) %>%
    summarise(
    mT = mean(Time2Incidence),
    mGA = mean(General_adherence_score.prior))

cor(df$Time2Incidence, df$General_adherence_score.prior, use = "complete.obs")

In [ ]:
df <- df %>%
    mutate(
    ancestry_pred = case_when(
    ancestry_pred %in% c("sas", "mid", "eas") ~ "Other",
    ancestry_pred %in% c("eur", "afr", "amr") ~ ancestry_pred, 
    TRUE ~ NA_character_),
    Education_quantitative = case_when(
        Education_level == "Less than a high school degree" ~ 1,
        Education_level == "Twelve Or GED" ~ 2,
        Education_level == "College Undergraduate" ~ 3,
        Education_level == "College Postgraduate" ~ 4,
        TRUE ~ NA_integer_
    )) %>%
    mutate(Income_quantitative =
          case_when(Income == ">= 200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "<=25k" ~ 1,
                   TRUE ~ NA_integer_)) %>%
    rename(CIDI_MDD = MDD_Incl)
table(df$ancestry_pred)

In [ ]:
library(tidyverse)
library(gtsummary)

# Prepare data for Table 1
# Assuming your data is in 'final' dataframe for All of Us and 'lifelines_data' for Lifelines

create_table1 <- function(data, cohort_name) {
  
  # Create table with demographics and outcomes
  table1 <- data %>%
    mutate(Income_quantitative = as.numeric(Income_quantitative),
          Education_quantitative = as.numeric(Education_quantitative),
          age_at_first_AD_unscaled = as.numeric(age_at_first_AD_unscaled),
          MDD_symptom_count_unscaled = as.numeric(MDD_symptom_count_unscaled)) %>%
    select(
      # Demographics
      age_at_first_AD_unscaled,
      sex_at_birth,
      ancestry_pred,
      Income_quantitative, 
      Education_quantitative,
      
      # Clinical characteristics
      Depression,
      CIDI_MDD,
      MDD_symptom_count_unscaled,
      Anxiety,
      # Add other comorbidities here
      
      # Treatment outcome
      FinalClassification.treatment
    ) %>%
    tbl_summary(
      by = FinalClassification.treatment, 
      label = list(
        age_at_first_AD_unscaled ~ "Age at first AD (years)",
        sex_at_birth ~ "Sex",
        ancestry_pred ~ "Ancestry",
        Depression ~ "Recorded Depression",
        CIDI_MDD ~ "MDD CIDI-sf",
        MDD_symptom_count_unscaled ~ "MDD CIDI-sf symptom count",
        Anxiety ~ "Anxiety disorder",
        Income_quantitative ~ "Income level",
        Education_quantitative ~ "Education level"
      ),
      type = list(Income_quantitative ~ "continuous",
                 Education_quantitative ~ "continuous",
                  MDD_symptom_count_unscaled ~ "continuous"),
      statistic = list(
        all_continuous() ~ "{mean} ({sd})",
        all_categorical() ~ "{n} ({p}%)"
      ),
      digits = all_continuous() ~ 1,
      missing = "no"
    ) %>%
    modify_header(label = paste0("**", cohort_name, "**"))
  
  return(table1)
}

# Create outcome characteristics table
create_outcome_characteristics <- function(data, cohort_name) {
  
  # Calculate time to outcome and adherence by outcome group
  outcome_stats <- data %>%
    group_by(FinalClassification.treatment) %>%
    summarise(
      N = n(),
      Time_mean = mean(Time2Incidence, na.rm = TRUE),  # replace with your actual variable name
      Time_sd = sd(Time2Incidence, na.rm = TRUE),
      Adherence_mean = mean(General_adherence_score.prior, na.rm = TRUE),  # replace with your actual variable name
      Adherence_sd = sd(General_adherence_score.prior, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    mutate(
      Time = sprintf("%.1f (%.1f)", Time_mean, Time_sd),
      Adherence = sprintf("%.2f (%.2f)", Adherence_mean, Adherence_sd),
      Cohort = cohort_name
    ) %>%
    select(Cohort, FinalClassification.treatment, N, Time, Adherence)
  
  return(outcome_stats)
}

# Generate tables for both cohorts
aou_table1 <- create_table1(df, "All of Us")

# Generate outcome characteristics
aou_outcomes <- create_outcome_characteristics(df, "All of Us")


In [ ]:
library(gt)

# Save demographics table as HTML (easiest to view)
aou_table1 %>%
  as_gt() %>%
  gtsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Table1_demographics_AoU.html")

# Save outcome characteristics
aou_outcomes_formatted <- aou_outcomes %>%
  mutate(
    FinalClassification.treatment = recode(FinalClassification.treatment,
                                          "Continuation" = "Sustained use",
                                          "Switching" = "Switching", 
                                          "Discontinuation" = "Discontinuation")
  ) %>%
  select(-Cohort) %>%
  gt() %>%
  cols_label(
    FinalClassification.treatment = "Treatment Outcome",
    N = "N",
    Time = "Time from first AD (years), M (SD)",
    Adherence = "Mean adherence (0-1), M (SD)"
  )

aou_outcomes_formatted %>%
  gtsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Table1_outcomes_AoU.html")